<a href="https://colab.research.google.com/github/shaktidharreddy/Agentic_AI_eduhubspot_aug_sep_2026/blob/main/Module00_Python_OpenAI/OpenAI_ResponsesAPI_ToolCalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
def get_weather(location):
    # Simple local execution mock representation for the get_weather tool
    return json.dumps({"status": "Sunny", "temp": "22C", "location": location})


# Shared tool specification array
my_tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current weather conditions for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name"}
            },
            "required": ["location"]
        }
    }
]

from openai import OpenAI
import json
from google.colab import userdata

# Retrieve the OpenAI API key securely from Colab Secrets
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception as e:
    print("Error retrieving secret. Please ensure 'OPENAI_API_KEY' is added to Colab Secrets.")
    OPENAI_API_KEY = None

client = OpenAI(api_key=OPENAI_API_KEY)



# 1. Initialize our conversation history
conversation_history = [
    {"role": "user", "content": "What's the weather like in Paris?"}
]

response = client.responses.create(
    model="gpt-4o",
    input=conversation_history,
    tools=my_tools
)

print(response)

Response(id='resp_0ac58d1dc2c719e9006b0a3e78065887d1a4f55cee84a846e6', created_at=1795833464.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-2024-08-06', object='response', output=[ResponseFunctionToolCall(arguments='{"location":"Paris"}', call_id='call_yWdEBCazw0lKUbRKp1Crj6sY', name='get_weather', type='function_call', id='fc_0ac58d1dc2c719e9006b0a3e78c29c87d1954534d9c54908f0', caller=None, namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'City name'}}, 'required': ['location'], 'additionalProperties': False}, strict=True, type='function', allowed_callers=None, defer_loading=None, description='Get current weather conditions for a location', output_schema=None)], top_p=1.0, background=False, completed_at=1795833464.0, conversation=None, max_output_tokens=None, max_too

In [7]:
# A temporary list to collect tool outputs before calling the LLM again
tool_outputs = []

# Step 2: Loop through the output items to find tool calls
for tool_call in response.output:
    if tool_call.type == "function_call":
        print(f"Model wants to call: {tool_call.name}")

        # Step 3: Run your local code execution
        if tool_call.name == "get_weather":
            args = json.loads(tool_call.arguments)
            tool_result = get_weather(**args)
        else:
            tool_result = "{\"status\": \"Unknown\"}"

        # Step 4: Keep track of this tool_call item and its corresponding output
        tool_outputs.append(tool_call)
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": tool_result
        })

# Step 5: If tools were called, extend the history and make a SINGLE call outside the loop
if tool_outputs:
    conversation_history.extend(tool_outputs)

    final_response = client.responses.create(
        model="gpt-4o",
        input=conversation_history
    )
    print(final_response.output_text)



Model wants to call: get_weather
The weather in Paris is currently sunny with a temperature of 22°C.


In [8]:


# second example

from openai import OpenAI
import json

client2 = OpenAI(api_key=OPENAI_API_KEY)

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },
            "required": ["sign"],
        },
    },
]

def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."

# Create a running input list we will add to over time
input_list = [
    {"role": "user", "content": "What is my horoscope? I am an Aquarius."}
]

# 2. Prompt the model with tools defined
response = client2.responses.create(
    model="gpt-5.4-nano",
    tools=tools,
    input=input_list,
)

# Save function call outputs for subsequent requests
input_list += response.output

def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    if name == "get_horoscope":
        return get_horoscope(**args)

for tool_call in response.output:
    if tool_call.type == "function_call":
        if tool_call.name == "get_horoscope":
            # 3. Execute the function logic for get_horoscope
            args = json.loads(tool_call.arguments)
            result = call_function(tool_call.name, args)

            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": str(result)
            })

print("Final input:")
print(input_list)

response = client2.responses.create(
    model="gpt-5.4-nano",
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final input:
[{'role': 'user', 'content': 'What is my horoscope? I am an Aquarius.'}, ResponseFunctionToolCall(arguments='{"sign":"Aquarius"}', call_id='call_7f247825b14448deb23c4f42409fbb39', name='get_horoscope', type='function_call', id='fc_0e8924ee55194643006b0a44f3d35487d1a41adc81eb9ac6f9', caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_7f247825b14448deb23c4f42409fbb39', 'output': 'Aquarius: Next Tuesday you will befriend a baby otter.'}]
Final output:
{
  "id": "resp_0e8924ee55194643006b0a44f4536887d1b00cac34e677a4df",
  "created_at": 1795835124.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a horoscope generated by a tool.",
  "metadata": {},
  "model": "gpt-5.4-nano-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0e8924ee55194643006b0a44f4b26487d1aec36e2e959c53df",
      "content": [
        {
          "annotations": [],
          "text": "Aquarius: Next Tuesd